In [7]:
%matplotlib qt
#plots open in a separate Qt window instead of inline in the notebook. Must run before pyplot is used.

import numpy as np
from scipy.stats import linregress
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

data_dir = Path("data_with_stress") #creates a Path object for the directory containing the data files

data = {}
elastic_strain = {}
elastic_stress = {}

fig, axes = plt.subplots(2, 7, figsize=(24, 8), constrained_layout=True) #one window: row 0 = A1-A7, row 1 = S1-S7
axes = axes.flatten() #2D grid of axes -> 1D list so axes[i] picks the i-th plot

for i, file_path in enumerate(sorted(data_dir.glob("*.csv"))): 
    """sorted() keeps the files in the order they are in the folder
    enumerate gives a counter i (0, 1, 2...) alongside each file
    file_path.stem defines name array to be A1,..., S7 without the .csv
    np.loadtxt reads a text file of numbers and returns a 2D numpy array""" 
    name = file_path.stem 
    table = np.loadtxt(file_path, delimiter=",", skiprows=1) 
    data[name] = table
    #this sets up NumPy arrays. Each data column can be called through e.g. data["S3"][:,4] gives the 5th columns in S3 (stress)


    force = table[:,1]
    strain100 = table[:,2]
    stress = table[:,4]

    ax = axes[i]
    ax.plot(strain100 / 100, stress)
    ax.set_title(f"{name} Stress vs. Strain")
    ax.set_xlabel("Strain (in/in)")
    ax.set_ylabel("Stress (psi)")
    

plt.show() #after the loop, so the window opens once with all 14 plots


In [8]:
#Ultimate tensile strength caclulation(max(stress))
uts = {}
for name, table in data.items():
    uts[name] = table[:,4].max() #stores each uts under its specimen's key
    print(f"{name}: UTS = {uts[name]:.3e}")

A1: UTS = 4.708e+04
A2: UTS = 4.705e+04
A3: UTS = 4.611e+04
A4: UTS = 6.274e+04
A5: UTS = 4.676e+04
A6: UTS = 4.657e+04
A7: UTS = 4.725e+04
S1: UTS = 1.354e+05
S2: UTS = 1.369e+05
S3: UTS = 1.343e+05
S4: UTS = 1.333e+05
S5: UTS = 1.367e+05
S6: UTS = 1.329e+05
S7: UTS = 1.362e+05


In [9]:
#Young's modulus calculation

E = {}
b = {}
r2 = {}
lo = 0.0002  #strain in in/in
hi = 0.002 #strain in in/in
 #0% and 0.2% can vary to find the linear portion
for name, table in data.items():
    """items() gives each key -> value pair e.g. "name" is A1 and table is uts 2D array.
    This jusr reuses what cell 0 loaded."""
    strain = table[:,2] / 100 #this specimen's strain, % -> in/in
    stress = table[:,4]       #this specimen's stress (psi)

    elastic = (strain > lo) & (strain < hi) #same mask on strain and stress so the points stay paired
    elastic_strain[name] = strain[elastic]
    elastic_stress[name] = stress[elastic]

    res = linregress(elastic_strain[name], elastic_stress[name])

    E[name] = res.slope #store by name so all 14 are kept, not just the last
    b[name] = res.intercept
    r2[name] = res.rvalue**2

    print(f"{name}: E = {E[name]:.3e} psi, b = {b[name]:.1f} psi, R² = {r2[name]:.4f}")

#looks like someone didn't mount their extensometer incorrectly for A4

lower = 0.16 / 100
upper = 0.2 / 100
A4_strain = (data["A4"][:,2] / 100)
A4_elastic = (A4_strain > lower) & (A4_strain < upper)
A4_eps = A4_strain[A4_elastic]
A4_stress = data["A4"][:,4][A4_elastic]

A4_res = linregress(A4_eps, A4_stress)
print(f"A4: A4E = {A4_res.slope:.3e} psi, A4b = {A4_res.intercept:.1f} psi , A4R² = {A4_res.rvalue ** 2:.4f}")
A4E = A4_res.slope
A4b = A4_res.intercept
print(E["A1"])

A1: E = 9.867e+06 psi, b = -22.1 psi, R² = 0.9993
A2: E = 1.026e+07 psi, b = -324.9 psi, R² = 0.9993
A3: E = 9.981e+06 psi, b = -124.6 psi, R² = 0.9996
A4: E = 1.429e+07 psi, b = -7339.1 psi, R² = 0.0842
A5: E = 9.153e+06 psi, b = -367.0 psi, R² = 0.9993
A6: E = 9.746e+06 psi, b = -91.8 psi, R² = 0.9989
A7: E = 1.019e+07 psi, b = -63.3 psi, R² = 0.9990
S1: E = 3.046e+07 psi, b = -682.7 psi, R² = 0.9998
S2: E = 3.126e+07 psi, b = -236.3 psi, R² = 0.9997
S3: E = 3.042e+07 psi, b = -1009.8 psi, R² = 0.9996
S4: E = 3.114e+07 psi, b = -130.8 psi, R² = 0.9997
S5: E = 3.009e+07 psi, b = -91.1 psi, R² = 0.9997
S6: E = 3.174e+07 psi, b = 334.1 psi, R² = 0.9999
S7: E = 3.096e+07 psi, b = -142.1 psi, R² = 0.9999
A4: A4E = 9.973e+06 psi, A4b = 88.8 psi , A4R² = 0.9995
9867321.106662787


In [10]:
#Yield Strength (where 0.2% offset intersects the main plot generate plots with both 0.2% offset and full curve)

#generate 0.2% offset plot

#add hashtag or remove for defective data--may be worth putting in report
E["A4"] = A4E 
b["A4"] = A4b
yield_02 = {}
yield_strain = {}

fig, axes = plt.subplots(2, 7, figsize=(24, 8), constrained_layout=True)
axes = axes.flatten()

for i, (name, table) in enumerate(data.items()):

    ax = axes[i] #pick this specimen's panel first, so the yield dot below lands on the right plot

    strain = table[:,2] / 100
    stress = table[:,4]

    offset = E[name] * (strain - 0.002) + b[name] 

    #yiled strength calculation
    i_uts = np.argmax(stress)
    s = stress[:i_uts + 1]
    e = strain[:i_uts + 1]
    off = offset[:i_uts + 1]

    diff = s - off 

    intersect = np.where((diff[:-1] > 0) & (diff[1:] <= 0))[0] #row k is + and row k+1 is negative or zero
    # diff[:-1] = rows 0..n-2, diff[1:] = rows 1..n-1 → compares each row with the next
    # & = elementwise AND on the two boolean arrays
    # np.where(bool_array)[0] → array of indices k where it's True

    if len(intersect) == 0:                       # no crossing found
        yield_02[name] = np.nan
        yield_strain[name] = np.nan
        print(f"{name}: no 0.2% offset crossing before UTS")
    else:
        k = intersect[0]                          # first crossing = yield
        # linear interpolation between rows k and k+1 to where diff = 0
        t = diff[k] / (diff[k] - diff[k + 1])            # fraction 0..1 of the way from k to k+1
        yield_02[name]     = s[k] + t * (s[k + 1] - s[k])
        yield_strain[name] = e[k] + t * (e[k + 1] - e[k])

        ax.plot(yield_strain[name], yield_02[name], "ro", label="0.2% yield") #red dot at yield: x = strain, y = stress, "ro" = red circle
        print(f"{name}: yield = {yield_02[name]:.3e} psi at ε = {yield_strain[name]:.5f}")


    ax.plot(strain, stress, label="Stress-Strain")
    ax.plot(strain, offset, "--", label="0.2% offset")

    ax.set_ylim(0, stress.max() * 1.1) #ylim is needed so offset does not make plot very large x1.1 makes it a teeny but above ult
    ax.set_title(f"{name} Stress-Strain curve")
    ax.set_xlabel("Strain (in/in)")
    ax.set_ylabel("Stress (psi)")
    ax.legend(fontsize=7)
plt.show()
    

A1: yield = 4.332e+04 psi at ε = 0.00639
A2: yield = 4.325e+04 psi at ε = 0.00625
A3: yield = 4.197e+04 psi at ε = 0.00622
A4: yield = 5.889e+04 psi at ε = 0.00790
A5: yield = 4.556e+04 psi at ε = 0.00702
A6: yield = 4.283e+04 psi at ε = 0.00640
A7: yield = 4.352e+04 psi at ε = 0.00628
S1: yield = 1.257e+05 psi at ε = 0.00615
S2: yield = 1.283e+05 psi at ε = 0.00611
S3: yield = 1.253e+05 psi at ε = 0.00615
S4: yield = 1.243e+05 psi at ε = 0.00600
S5: yield = 1.285e+05 psi at ε = 0.00627
S6: yield = 1.236e+05 psi at ε = 0.00588
S7: yield = 1.270e+05 psi at ε = 0.00611


In [17]:
#Yield Strength
#same as the cell above, but 14 separate windows (one plot each) instead of one window with 14 subplots opens

#generate 0.2% offset plot

#add hashtag or remove for defective data--may be worth putting in report
E["A4"] = A4E 
b["A4"] = A4b
yield_02 = {}
yield_strain = {}

plot_dir = Path("plots")
plot_dir.mkdir(exist_ok=True)

for name, table in data.items(): #no enumerate: there's no axes[i] to pick, so the counter i isn't needed

    fig, ax = plt.subplots(figsize=(6, 4)) #inside the loop -> a new window each pass; no grid arguments, so ax is one axes object

    strain = table[:,2] / 100
    stress = table[:,4]

    offset = E[name] * (strain - 0.002) + b[name] 

    #yiled strength calculation
    i_uts = np.argmax(stress)
    s = stress[:i_uts + 1]
    e = strain[:i_uts + 1]
    off = offset[:i_uts + 1]

    diff = s - off 

    intersect = np.where((diff[:-1] > 0) & (diff[1:] <= 0))[0] #row k is + and row k+1 is negative or zero
    # diff[:-1] = rows 0..n-2, diff[1:] = rows 1..n-1 → compares each row with the next
    # & = elementwise AND on the two boolean arrays
    # np.where(bool_array)[0] → array of indices k where it's True

    if len(intersect) == 0:                       # no crossing found
        yield_02[name] = np.nan
        yield_strain[name] = np.nan
        print(f"{name}: no 0.2% offset crossing before UTS")
    else:
        k = intersect[0]                          # first crossing = yield
        # linear interpolation between rows k and k+1 to where diff = 0
        t = diff[k] / (diff[k] - diff[k + 1])            # fraction 0..1 of the way from k to k+1
        yield_02[name]     = s[k] + t * (s[k + 1] - s[k])
        yield_strain[name] = e[k] + t * (e[k + 1] - e[k])

        ax.plot(yield_strain[name], yield_02[name], "ro", label="0.2% yield") #red dot at yield: x = strain, y = stress, "ro" = red circle
        print(f"{name}: yield = {yield_02[name]:.3e} psi at ε = {yield_strain[name]:.5f}")


    ax.plot(strain, stress, label="Stress-Strain")
    ax.plot(strain, offset, "--", label="0.2% offset")

    ax.set_ylim(0, stress.max() * 1.1) #ylim is needed so offset does not make plot very large x1.1 makes it a teeny but above ult
    ax.set_title(f"{name} Stress-Strain curve")
    ax.set_xlabel("Strain (in/in)")
    ax.set_ylabel("Stress (psi)")
    ax.legend(fontsize=7)
    fig.savefig(plot_dir / f"{name}_yield.png", dpi=200, bbox_inches="tight")
plt.show() #still after the loop: all 14 windows open at once
    

A1: yield = 4.332e+04 psi at ε = 0.00639
A2: yield = 4.325e+04 psi at ε = 0.00625
A3: yield = 4.197e+04 psi at ε = 0.00622
A4: yield = 5.889e+04 psi at ε = 0.00790
A5: yield = 4.556e+04 psi at ε = 0.00702
A6: yield = 4.283e+04 psi at ε = 0.00640
A7: yield = 4.352e+04 psi at ε = 0.00628
S1: yield = 1.257e+05 psi at ε = 0.00615
S2: yield = 1.283e+05 psi at ε = 0.00611
S3: yield = 1.253e+05 psi at ε = 0.00615
S4: yield = 1.243e+05 psi at ε = 0.00600
S5: yield = 1.285e+05 psi at ε = 0.00627
S6: yield = 1.236e+05 psi at ε = 0.00588
S7: yield = 1.270e+05 psi at ε = 0.00611


In [18]:
#close every plot window (with %matplotlib qt each window stays open until closed)
plt.close("all")

In [16]:
#Two dataframes to be exported as a csv one for material A and one for material S

#Column headers Diameter before test, "d_o (in)"; Area before test, "A_o in^2", Young's modulus, "E Msi"; r-squared, R^2; yield strength, "Yield ksi" ; ultimate tensile strngth, "UTS ksi"; 
# Rows A1, ..., A7 

#same column headers
#Rows S1, ..., S7

#add hashtag or remove for defective data--same toggle as E["A4"] and b["A4"] in cell 3
r2["A4"] = A4_res.rvalue**2

#diameters typed into stres_clalc.ipynb, which saves them to diameters.csv
diam = pd.read_csv("diameters.csv", index_col="Specimen")   #index_col: the Specimen column becomes the index (A1 ... S7)
d_o = diam["d_o (in)"]         #a Series indexed by specimen, so it lines up by key like the dicts below
A_o = np.pi * d_o**2 / 4       #area before test (in^2), vectorized over all 14

#dicts keyed by specimen -> rows are matched by key (A1 ... S7 become the index)
#pd.Series(E) / 1e6 turns the dict into a Series so the whole column can be divided at once (psi -> Msi)
results = pd.DataFrame({
    "d₀ (in)":   d_o,
    "A₀ (in²)":  A_o,
    "E (Msi)":   pd.Series(E) / 1e6, #psi -> Msi
    "R²":        r2,
    "Yield (ksi)": pd.Series(yield_02) / 1e3,  #psi -> ksi
    "UTS (ksi)":   pd.Series(uts) / 1e3, 
})
#the number is the number of decimal places
results = results.round({"d₀ (in)": 4, "A₀ (in²)": 4, "E (Msi)": 4, "R²": 4, "Yield (ksi)": 4, "UTS (ksi)": 4})

results.index.name = "Specimen"
results = results.reset_index()    #index -> normal "Specimen" column

#split by the first letter of the specimen name
results_A = results[results["Specimen"].str.startswith("A")]
results_S = results[results["Specimen"].str.startswith("S")]

#encoding="utf-8-sig" marks the file as UTF-8 so Minitab/Excel show ₀ and ² correctly instead of garbled characters
results_A.to_csv("results_A.csv", index=False, encoding="utf-8-sig")   #index=False: no extra row-number column
results_S.to_csv("results_S.csv", index=False, encoding="utf-8-sig")

print(results_A)
print()
print(results_S)


  Specimen  d₀ (in)  A₀ (in²)  E (Msi)      R²  Yield (ksi)  UTS (ksi)
0       A1    0.499    0.1956   9.8673  0.9993      43.3219    47.0790
1       A2    0.502    0.1979  10.2565  0.9993      43.2472    47.0485
2       A3    0.500    0.1963   9.9807  0.9996      41.9746    46.1116
3       A4    0.498    0.1948   9.9734  0.9995      58.8864    62.7368
4       A5    0.502    0.1979   9.1525  0.9993      45.5557    46.7554
5       A6    0.503    0.1987   9.7456  0.9989      42.8281    46.5748
6       A7    0.501    0.1971  10.1871  0.9990      43.5199    47.2466

   Specimen  d₀ (in)  A₀ (in²)  E (Msi)      R²  Yield (ksi)  UTS (ksi)
7        S1   0.4950    0.1924  30.4587  0.9998     125.7401   135.3653
8        S2   0.4990    0.1956  31.2602  0.9997     128.3461   136.8855
9        S3   0.4980    0.1948  30.4193  0.9996     125.3390   134.3041
10       S4   0.5005    0.1967  31.1425  0.9997     124.3250   133.2707
11       S5   0.4960    0.1932  30.0864  0.9997     128.4543   136.7350